# Lab 3.1 - Fine-Tune BERT for Sentiment (Solution)
**Day 3 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/lab3_1_bert_finetune/solution/lab3_1_bert_finetune.ipynb)

**Goal:** Take a pre-trained DistilBERT encoder, add a classification head,
and fine-tune it on a tiny labeled sentiment dataset with the Hugging Face
`Trainer`. Compare predictions before and after training.

This solution has the full working code. It runs on a Colab CPU runtime in a
few minutes; no GPU needed. Summarization is covered as a concept at the end.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q transformers torch

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

torch.manual_seed(42)
MODEL_NAME = "distilbert-base-uncased"

# A tiny labeled dataset: 1 = positive, 0 = negative.
TRAIN_TEXTS = [
    "This product is fantastic and works perfectly.",
    "Absolutely love it, best purchase this year.",
    "Great quality and fast shipping, highly recommend.",
    "The app is smooth and easy to use.",
    "Wonderful experience, I am very happy.",
    "Excellent support and a great result.",
    "This is amazing, exceeded my expectations.",
    "Very reliable and worth every penny.",
    "Clean design and it just works.",
    "Superb value, I would buy again.",
    "Terrible experience, it broke on day one.",
    "I hate this, complete waste of money.",
    "Awful quality and slow shipping.",
    "The app keeps crashing and is frustrating.",
    "Very disappointing, I want a refund.",
    "Poor support and a useless product.",
    "This is horrible, do not buy it.",
    "Cheaply made and stopped working fast.",
    "Buggy and confusing to use.",
    "Worst purchase I have made all year.",
]
TRAIN_LABELS = [1] * 10 + [0] * 10

EVAL_TEXTS = [
    "I really enjoyed using this, it works great.",
    "A delightful and reliable little tool.",
    "This was a frustrating and buggy mess.",
    "Do not waste your money on this.",
]
EVAL_LABELS = [1, 1, 0, 0]

TEST_SENTENCES = [
    "I love how well this works!",
    "This is the worst thing I have ever bought.",
]

## 1. Load the pre-trained encoder with a fresh head
`distilbert-base-uncased` ships without a classification head, so
`AutoModelForSequenceClassification` adds a new, randomly initialized one. The
encoder already understands English; only the head needs training.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)
print("Loaded", MODEL_NAME, "with a fresh 2-class head.")

## 2. Predict before fine-tuning
The head is random, so predictions are essentially a coin flip. This is the
baseline we will improve.

In [ ]:
def predict(sentences):
    model.eval()
    enc = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)
    preds = probs.argmax(dim=-1)
    for sent, pred, prob in zip(sentences, preds, probs):
        label = model.config.id2label[pred.item()]
        print(f"  [{label:>8}  p={prob[pred].item():.2f}]  {sent}")


print("Before fine-tuning:")
predict(TEST_SENTENCES)

## 3. Wrap the data for the Trainer
The `Trainer` expects a dataset that returns tokenized tensors plus a
`labels` field. We tokenize up front and store everything in a small
`Dataset`.

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(texts, padding=True, truncation=True, max_length=64)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_ds = SentimentDataset(TRAIN_TEXTS, TRAIN_LABELS)
eval_ds = SentimentDataset(EVAL_TEXTS, EVAL_LABELS)
print(f"Train examples: {len(train_ds)}, eval examples: {len(eval_ds)}")

## 4. Fine-tune with the Trainer
A few epochs over 20 examples is enough for this toy task. `use_cpu=True`
keeps the run identical on Colab CPU and a laptop. The `Trainer` handles the
training loop, batching, and evaluation.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}


args = TrainingArguments(
    output_dir="bert-sentiment",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    eval_strategy="epoch",
    logging_steps=5,
    report_to="none",
    use_cpu=True,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
print("\nEval:", trainer.evaluate())

## 5. Predict after fine-tuning
The same encoder now drives a trained head. The two test sentences should
come back with the correct label and high confidence.

In [ ]:
print("After fine-tuning:")
predict(TEST_SENTENCES)

## 6. Summarization (concept)
Sentiment is a classification task: an encoder plus a small head. Text
summarization is a sequence-to-sequence task, so it uses an encoder-decoder
model such as T5 or BART rather than an encoder-only model like BERT. The
fine-tuning recipe is the same shape: load a pre-trained checkpoint, feed it
labeled examples (article plus reference summary), and train with the
`Trainer` (using `Seq2SeqTrainer`). We keep summarization as a concept here so
the hands-on time stays on one clean fine-tune. The first stretch goal shows a
pre-trained summarizer running inference, no training required.

## Stretch goals
1. Run a pre-trained summarizer with a pipeline (inference only):
   `from transformers import pipeline;
   summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6")`
   then summarize a paragraph. Note the extra model download.
2. Add a few of your own labeled sentences to the training data, retrain, and
   check whether a tricky test sentence flips to the right label.
3. Freeze the encoder (set `param.requires_grad = False` for
   `model.distilbert` parameters) and train only the head. Compare speed and
   accuracy against full fine-tuning.

In [ ]:
# Stretch 3: feature extraction (train only the head).
frozen_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)
for param in frozen_model.distilbert.parameters():
    param.requires_grad = False
trainable = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in frozen_model.parameters())
print(f"Feature extraction trains {trainable:,} of {total:,} parameters "
      f"({100 * trainable / total:.2f}%).")

## Expected output
- Before fine-tuning, the two test sentences are labeled near random (about
  0.5 either way).
- Training loss drops over the epochs and eval accuracy reaches 1.0 on the
  tiny eval set.
- After fine-tuning, the positive sentence is labeled positive and the
  negative one negative, with clearly higher confidence than before.
- The feature-extraction stretch reports that only about 0.8% of parameters
  (the head) are trainable when the encoder is frozen.